# Deploy de Modelos PyTorch - Versao Local

## Visao Geral

Este notebook demonstra o fluxo completo de criacao, treinamento, serializacao e deploy de um modelo **PyTorch**, adaptado para execucao local.

> **Nota:** Esta versao usa PyTorch em vez de TensorFlow por compatibilidade com Python 3.14.

### Etapas do Processo:
1. **Preparar** - Criar os artefatos do modelo necessarios para o deploy
2. **Verificar** - Simular chamadas ao modelo antes do deploy
3. **Salvar** - Serializar o modelo para disco
4. **Carregar** - Restaurar o modelo salvo
5. **Predizer** - Realizar inferencias com o modelo

---

## Conteudo

- Introducao  
  - Dataset Fashion-MNIST
- Criar um Modelo PyTorch
- Serializacao do Modelo
- Carregar e Verificar o Modelo
- Realizar Predicoes
- Criar API REST Local (Flask)

In [ ]:
# Importacoes principais
import os
import json
import logging
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

# Configuracao de logging e warnings
logging.basicConfig(format='%(levelname)s:%(message)s', level=logging.ERROR)
warnings.filterwarnings('ignore')

# Verificar versoes
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA disponivel: {torch.cuda.is_available()}")

# Introducao - Dataset Fashion-MNIST

O **Fashion_MNIST** e um conjunto de dados de imagens de artigos de vestuario da Zalando:
- **60.000** exemplos de treino e **10.000** de teste
- Imagens em escala de cinza de **28x28 pixels**
- **10 classes** de roupas e acessorios

| Label | Descricao |
|-------|-----------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
# Nomes das classes
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Transformacoes para normalizar os dados
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normaliza para [-1, 1]
])

# Carregar dataset Fashion-MNIST
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

# Criar DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Tamanho do conjunto de treino: {len(train_dataset)}")
print(f"Tamanho do conjunto de teste: {len(test_dataset)}")
print(f"Formato de uma imagem: {train_dataset[0][0].shape}")

In [ ]:
# Visualizar algumas imagens do conjunto de treino
from matplotlib import pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]
    # Desnormalizar para visualizacao
    image = image.squeeze().numpy() * 0.5 + 0.5
    ax.imshow(image, cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')

plt.suptitle('Exemplos do Dataset Fashion-MNIST', fontsize=14)
plt.tight_layout()
plt.show()

## Criacao do Modelo PyTorch

Arquitetura da rede neural:
- **Camada de entrada**: Flatten (28x28 = 784 nos)
- **Camada oculta**: Linear com 128 nos + ReLU
- **Dropout**: 20% para regularizacao
- **Camada de saida**: Linear com 10 nos (uma para cada classe)

In [ ]:
# Definicao do modelo
class FashionMNISTModel(nn.Module):
    def __init__(self):
        super(FashionMNISTModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Criar instancia do modelo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FashionMNISTModel().to(device)

print(f"Dispositivo: {device}")
print(f"\nArquitetura do modelo:")
print(model)

## Treinamento do Modelo

In [ ]:
# Funcao de treinamento
def train_model(model, train_loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    history = {'loss': [], 'accuracy': []}
    
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Estatisticas
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_acc = correct / total
        history['loss'].append(epoch_loss)
        history['accuracy'].append(epoch_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.4f}")
    
    return history

# Treinar o modelo
print("Iniciando treinamento...")
history = train_model(model, train_loader, epochs=5)
print("\nTreinamento concluido!")

In [ ]:
# Avaliar o modelo no conjunto de teste
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    return accuracy

test_accuracy = evaluate_model(model, test_loader)
print(f"Acuracia no conjunto de teste: {test_accuracy:.4f}")

## Serializacao do Modelo PyTorch

O PyTorch oferece dois formatos principais para salvar modelos:

1. **State Dict** (recomendado): Salva apenas os pesos do modelo
2. **Modelo Completo**: Salva a estrutura e pesos (usando pickle)

### Arquivos gerados:
- **model.pth**: Pesos do modelo (state_dict)
- **model_full.pth**: Modelo completo
- **metadata.json**: Metadados do modelo
- **score.py**: Script de inferencia

In [ ]:
# Criar diretorio para artefatos
artifact_dir = Path("./model_artifacts_pytorch")
artifact_dir.mkdir(exist_ok=True)

print(f"Diretorio de artefatos: {artifact_dir.absolute()}")

In [ ]:
class LocalPyTorchModel:
    """
    Classe para gerenciar modelos PyTorch localmente,
    similar ao TensorFlowModel do OCI ADS.
    """
    
    def __init__(self, estimator, artifact_dir, model_class=None):
        self.estimator = estimator
        self.artifact_dir = Path(artifact_dir)
        self.model_class = model_class or type(estimator)
        self.artifact_dir.mkdir(exist_ok=True)
        
        self._status = {
            'initiate': {'status': 'Done', 'details': 'Model initialized'},
            'prepare': {'status': 'Pending', 'details': ''},
            'verify': {'status': 'Pending', 'details': ''},
            'save': {'status': 'Pending', 'details': ''},
            'deploy': {'status': 'Pending', 'details': ''},
        }
        
        self.metadata = {}
        self.input_schema = None
        self.output_schema = None
        
    def summary_status(self):
        """Retorna DataFrame com status do processo"""
        data = []
        for step, info in self._status.items():
            data.append({
                'Step': step.capitalize(),
                'Status': info['status'],
                'Details': info['details']
            })
        return pd.DataFrame(data)
    
    def prepare(self, X_sample=None, y_sample=None, use_case_type=None):
        """Prepara os artefatos do modelo"""
        try:
            # Salvar state_dict (recomendado)
            torch.save(self.estimator.state_dict(), self.artifact_dir / 'model.pth')
            
            # Salvar modelo completo (alternativo)
            torch.save(self.estimator, self.artifact_dir / 'model_full.pth')
            
            # Criar schemas
            if X_sample is not None:
                self.input_schema = {
                    'dtype': str(X_sample.dtype) if hasattr(X_sample, 'dtype') else 'float32',
                    'shape': list(X_sample.shape) if hasattr(X_sample, 'shape') else [28, 28],
                }
                with open(self.artifact_dir / 'input_schema.json', 'w') as f:
                    json.dump(self.input_schema, f, indent=2)
            
            if y_sample is not None:
                unique_classes = np.unique(y_sample) if hasattr(y_sample, '__iter__') else list(range(10))
                self.output_schema = {
                    'num_classes': len(unique_classes),
                    'classes': list(map(int, unique_classes)),
                }
                with open(self.artifact_dir / 'output_schema.json', 'w') as f:
                    json.dump(self.output_schema, f, indent=2)
            
            # Criar runtime.yaml
            import yaml
            runtime_info = {
                'MODEL_ARTIFACT_VERSION': '1.0',
                'FRAMEWORK': 'PyTorch',
                'FRAMEWORK_VERSION': torch.__version__,
            }
            with open(self.artifact_dir / 'runtime.yaml', 'w') as f:
                yaml.dump(runtime_info, f)
            
            # Criar score.py
            score_content = '''"""Score script para inferencia PyTorch."""
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# Definicao do modelo (deve ser igual ao usado no treinamento)
class FashionMNISTModel(nn.Module):
    def __init__(self):
        super(FashionMNISTModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = None

def load_model(model_dir=None):
    """Carrega o modelo do disco."""
    global model
    if model_dir is None:
        model_dir = Path(__file__).parent
    else:
        model_dir = Path(model_dir)
    
    model = FashionMNISTModel()
    model.load_state_dict(torch.load(model_dir / "model.pth", weights_only=True))
    model.eval()
    return model

def predict(data, model=None):
    """Realiza predicao."""
    if model is None:
        model = load_model()
    
    if isinstance(data, list):
        data = np.array(data)
    if isinstance(data, np.ndarray):
        data = torch.tensor(data, dtype=torch.float32)
    
    with torch.no_grad():
        outputs = model(data)
        probabilities = torch.softmax(outputs, dim=1)
    
    return probabilities.numpy().tolist()

if __name__ == "__main__":
    model = load_model()
    print("Modelo carregado com sucesso!")
'''
            with open(self.artifact_dir / 'score.py', 'w') as f:
                f.write(score_content)
            
            # Metadata
            self.metadata = {
                'model_name': 'PyTorchModel',
                'framework': 'PyTorch',
                'framework_version': torch.__version__,
                'use_case_type': use_case_type or 'classification',
                'created_at': datetime.now().isoformat(),
            }
            with open(self.artifact_dir / 'metadata.json', 'w') as f:
                json.dump(self.metadata, f, indent=2)
            
            self._status['prepare'] = {
                'status': 'Done',
                'details': f'Artifacts saved to {self.artifact_dir}'
            }
            
            print(f"Artefatos preparados em: {self.artifact_dir}")
            print(f"Arquivos: {[f.name for f in self.artifact_dir.iterdir()]}")
            
        except Exception as e:
            self._status['prepare'] = {'status': 'Error', 'details': str(e)}
            raise
        
        return self
    
    def verify(self, X_test):
        """Verifica o modelo carregando do disco e fazendo predicao"""
        try:
            import importlib.util
            spec = importlib.util.spec_from_file_location("score", self.artifact_dir / 'score.py')
            score_module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(score_module)
            
            loaded_model = score_module.load_model(str(self.artifact_dir))
            predictions = score_module.predict(X_test, loaded_model)
            
            self._status['verify'] = {
                'status': 'Done',
                'details': f'Verified with {len(predictions)} predictions'
            }
            
            return predictions
            
        except Exception as e:
            self._status['verify'] = {'status': 'Error', 'details': str(e)}
            raise
    
    def save(self, display_name=None):
        """Salva metadados finais e retorna ID"""
        import hashlib
        import time
        
        model_id = hashlib.md5(f"{display_name}_{time.time()}".encode()).hexdigest()[:16]
        
        self.metadata['display_name'] = display_name or 'Unnamed Model'
        self.metadata['model_id'] = model_id
        
        with open(self.artifact_dir / 'metadata.json', 'w') as f:
            json.dump(self.metadata, f, indent=2)
        
        self._status['save'] = {'status': 'Done', 'details': f'Model ID: {model_id}'}
        
        print(f"Modelo salvo!")
        print(f"  Display Name: {display_name}")
        print(f"  Model ID: {model_id}")
        print(f"  Location: {self.artifact_dir}")
        
        return model_id

In [ ]:
# Criar objeto LocalPyTorchModel
pytorch_model = LocalPyTorchModel(estimator=model, artifact_dir=artifact_dir)

# Status inicial
pytorch_model.summary_status()

## Prepare

In [ ]:
# Preparar amostra de dados para schemas
sample_image, sample_label = test_dataset[0]

# Preparar artefatos
pytorch_model.prepare(
    X_sample=sample_image,
    y_sample=np.array([l for _, l in test_dataset]),
    use_case_type='MULTINOMIAL_CLASSIFICATION'
)

In [ ]:
# Status apos prepare
pytorch_model.summary_status()

## Verify

In [ ]:
# Preparar dados de teste
test_images = torch.stack([test_dataset[i][0] for i in range(3)])

# Verificar modelo
predictions = pytorch_model.verify(test_images)

print("Predicoes verificadas:")
for i, pred in enumerate(predictions):
    predicted_class = np.argmax(pred)
    real_label = test_dataset[i][1]
    print(f"  Exemplo {i}: Predito={class_names[predicted_class]}, Real={class_names[real_label]}")

## Save

In [ ]:
# Salvar modelo
model_id = pytorch_model.save(display_name="Demo FMNIST PyTorch Model")

In [ ]:
# Status final
pytorch_model.summary_status()

## Deploy - API REST com Flask

In [ ]:
# Criar API Flask para deploy local
flask_code = '''"""API REST para modelo PyTorch."""
from flask import Flask, request, jsonify
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

app = Flask(__name__)

class FashionMNISTModel(nn.Module):
    def __init__(self):
        super(FashionMNISTModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = None
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

def load_model():
    global model
    model = FashionMNISTModel()
    model.load_state_dict(torch.load("model.pth", weights_only=True))
    model.eval()
    return model

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model_loaded": model is not None})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.json.get("data")
        if data is None:
            return jsonify({"error": "Campo data nao encontrado"}), 400
        
        input_data = np.array(data, dtype=np.float32)
        if input_data.max() > 1:
            input_data = input_data / 255.0
        
        tensor_data = torch.tensor(input_data)
        
        with torch.no_grad():
            outputs = model(tensor_data)
            probs = torch.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1)
        
        return jsonify({
            "predictions": probs.numpy().tolist(),
            "classes": predicted.numpy().tolist(),
            "class_names": [class_names[c] for c in predicted.numpy()]
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    load_model()
    print("Servidor iniciado em http://localhost:5000")
    app.run(host="0.0.0.0", port=5000, debug=True)
'''

# Salvar API
with open(artifact_dir / "app.py", "w") as f:
    f.write(flask_code)

print(f"API salva em: {artifact_dir / 'app.py'}")
print(f"\\nPara executar:")
print(f"  cd {artifact_dir}")
print(f"  python app.py")